# Phase 1.3 — Clean & Save to Parquet
Reads the raw CSV, normalizes columns, builds the `text_for_embedding` field, and saves `products.parquet`.

## Task 1.3.1 — Select and rename columns

In [1]:
import pandas as pd

df_raw = pd.read_csv('../data/raw/ecommerce_dataset.csv')

# Map raw column names → our schema names
df = df_raw[[
    'product_id',
    'title',
    'brand',
    'category',
    'price_current',
    'rating_score',
    'description',
    'availability',
    'source',
]].rename(columns={
    'product_id':   'id',
    'price_current':'price',
    'rating_score': 'rating',
    'description':  'features',
})

# Add ingredients column (not in this dataset — kept for schema consistency)
df['ingredients'] = None

print('Shape:', df.shape)
df.head(3)

Shape: (1000, 10)


,id,title,brand,category,price,rating,features,availability,source,ingredients
0,9efc81ac-b34e-487a-b7a1-6fe9c9d7920b,Huawei Tablet Pro,Huawei,Electronics,1487.49,4.8,Crafted with attention to detail and backed by...,in_stock,amazon,None
1,6c26ff34-72d3-45d8-aa01-71e58253e3c6,Adidas Portable Hoodie Plus,Adidas,Fashion,1125.26,3.7,A versatile product suitable for a wide range ...,out_of_stock,amazon,None
2,b271d64f-790b-4561-af51-5ff5d7b0a132,Calvin Klein Sneakers Pro,Calvin,Fashion,1357.13,3.9,Thoughtfully designed to deliver value without...,out_of_stock,amazon,None


## Task 1.3.2 — Drop rows missing title or price

In [2]:
before = len(df)
df = df.dropna(subset=['title', 'price'])
after = len(df)
print(f'Dropped {before - after} rows. Remaining: {after}')

Dropped 0 rows. Remaining: 1000


## Task 1.3.3 — Normalize price to float
Price is already a float in this dataset. We still round it for cleanliness and sanity-check the range.

In [3]:
df['price'] = df['price'].astype(float).round(2)

print('Price range: ${:.2f} — ${:.2f}'.format(df['price'].min(), df['price'].max()))
print('Median price: ${:.2f}'.format(df['price'].median()))

Price range: $1.08 — $49720.60
Median price: $1723.33


## Task 1.3.5 — Build `text_for_embedding`
This is the text the embedding model will encode. More context = better search results.

We concatenate: title + brand + category + features (description).

In [4]:
def build_text(row):
    parts = [
        str(row['title'])    if pd.notna(row['title'])    else '',
        str(row['brand'])    if pd.notna(row['brand'])    else '',
        str(row['category']) if pd.notna(row['category']) else '',
        str(row['features']) if pd.notna(row['features']) else '',
    ]
    return ' | '.join(p for p in parts if p)

df['text_for_embedding'] = df.apply(build_text, axis=1)

# Preview a few
for text in df['text_for_embedding'].sample(3, random_state=1).values:
    print(text[:200])
    print()

Nestle Advanced Green Tea Lite | Nestle | Food | High-quality construction ensures long-term reliability and consistent performance.

Ralph Lauren Pro Jeans 2024 | Ralph | Fashion | A versatile product suitable for a wide range of applications and users.

Black+Decker Pruning Shears Plus | Black+Decker | Garden | Compact form, powerful results — an ideal solution for modern lifestyles.



## Task 1.3.6 — Save to Parquet
Parquet is a compressed column-based format — much faster to load than CSV for large datasets.

In [5]:
output_path = '../data/processed/products.parquet'
df.to_parquet(output_path, index=False)
print(f'Saved {len(df)} rows to {output_path}')

# Verify it round-trips correctly
verify = pd.read_parquet(output_path)
print('Verified shape:', verify.shape)
print('Columns:', list(verify.columns))
verify.head(3)

Saved 1000 rows to ../data/processed/products.parquet
Verified shape: (1000, 11)
Columns: ['id', 'title', 'brand', 'category', 'price', 'rating', 'features', 'availability', 'source', 'ingredients', 'text_for_embedding']


,id,title,brand,category,price,rating,features,availability,source,ingredients,text_for_embedding
0,9efc81ac-b34e-487a-b7a1-6fe9c9d7920b,Huawei Tablet Pro,Huawei,Electronics,1487.49,4.8,Crafted with attention to detail and backed by...,in_stock,amazon,None,Huawei Tablet Pro | Huawei | Electronics | Cra...
1,6c26ff34-72d3-45d8-aa01-71e58253e3c6,Adidas Portable Hoodie Plus,Adidas,Fashion,1125.26,3.7,A versatile product suitable for a wide range ...,out_of_stock,amazon,None,Adidas Portable Hoodie Plus | Adidas | Fashion...
2,b271d64f-790b-4561-af51-5ff5d7b0a132,Calvin Klein Sneakers Pro,Calvin,Fashion,1357.13,3.9,Thoughtfully designed to deliver value without...,out_of_stock,amazon,None,Calvin Klein Sneakers Pro | Calvin | Fashion |...
